In [1]:
!pip install -q -U transformers accelerate peft bitsandbytes datasets sentencepiece sacrebleu evaluate


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 86.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 39.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 112.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 14.4 MB/s eta 0:00:00


In [2]:
import torch, random, json, re
from datasets import load_dataset, Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, get_cosine_schedule_with_warmup
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
import sacrebleu

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU - switch runtime to T4/L4")


Tesla T4


In [3]:
def load_with_parquet_fallback(dataset_id):
    try:
        return load_dataset(dataset_id)
    except RuntimeError as e:
        if "scripts are no longer supported" in str(e):
            print(f"'{dataset_id}' uses a legacy loading script - falling back to its Parquet mirror.")
            return load_dataset(dataset_id, revision="refs/convert/parquet")
        raise

raw = load_with_parquet_fallback("rahular/itihasa")
print(raw)
for ex in raw["train"].select(range(3)):
    print(ex)
    print("---")


README.md:   0%|          | 0.00/3.03k [00:00<?, ?B/s]

itihasa.py:   0%|          | 0.00/4.89k [00:00<?, ?B/s]

'rahular/itihasa' uses a legacy loading script - falling back to its Parquet mirror.


Itihasa/train/0000.parquet: reconstructing file:   0%|          |  0.00B / 16.4MB            

Itihasa/train/0000.parquet: downloading bytes:           |  0.00B            

...

Itihasa/test/0000.parquet: downloading bytes:           |  0.00B            

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['translation'],
        num_rows: 75162
    })
    validation: Dataset({
        features: ['translation'],
        num_rows: 6149
    })
    test: Dataset({
        features: ['translation'],
        num_rows: 11722
    })
})
{'translation': {'en': 'The ascetic Vālmīki asked Nārada, the best of sages and foremost of those conversant with words, ever engaged in austerities and Vedic studies.', 'sn': 'ॐ तपः स्वाध्यायनिरतं तपस्वी वाग्विदां वरम्। नारदं परिपप्रच्छ वाल्मीकिर्मुनिपुङ्गवम्॥'}}
---
{'translation': {'en': 'Who at present in this world is like crowned with qualities, and with prowess, knowing duty, and grateful, and truthful, and firm in vow.', 'sn': 'कोन्वस्मिन् साम्प्रतं लोके गुणवान् कश्च वीर्यवान्। धर्मज्ञश्च कृतज्ञश्च सत्यवाक्यो दृढत्नतः॥'}}
---
{'translation': {'en': 'Who is qualified by virtue of his character, and who is engaged in the welfare of all creatures? Who is learned and capable. Who alone is ever lovely to be

In [4]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
first_ex = raw["train"][0]
if "translation" in first_ex and isinstance(first_ex["translation"], dict):
    sample_sa = first_ex["translation"]["sn"]
    sample_en = first_ex["translation"]["en"]
else:
    sample_sa = first_ex["sn"]
    sample_en = first_ex["en"]

sa_tokens = tokenizer.tokenize(sample_sa)
en_tokens = tokenizer.tokenize(sample_en)

print("Sanskrit text:", sample_sa)
print("Sanskrit tokens (%d):" % len(sa_tokens), sa_tokens)
print()
print("English text:", sample_en)
print("English tokens (%d):" % len(en_tokens), en_tokens)

sa_chars = len(sample_sa.replace(" ", ""))
en_chars = len(sample_en.replace(" ", ""))
print(f"\nFragmentation ratio - Sanskrit: {len(sa_tokens)/max(sa_chars,1):.2f} tok/char, "
      f"English: {len(en_tokens)/max(en_chars,1):.2f} tok/char")
print("Higher ratio = more subword fragmentation for that script (expected for Sanskrit with a")
print("primarily English/Chinese-trained BPE tokenizer). This motivates keeping generations short")
print("and using low-ish max_new_tokens, and is worth calling out explicitly in the report.")


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Sanskrit text: ॐ तपः स्वाध्यायनिरतं तपस्वी वाग्विदां वरम्। नारदं परिपप्रच्छ वाल्मीकिर्मुनिपुङ्गवम्॥
Sanskrit tokens (83): ['à¥Ĳ', 'Ġà¤', '¤', 'à¤ª', 'à¤', 'ĥ', 'Ġà¤¸', 'à¥įà¤', 'µ', 'à¤¾à¤', '§', 'à¥įà¤', '¯', 'à¤¾à¤', '¯', 'à¤¨', 'à¤¿à¤', '°', 'à¤¤', 'à¤Ĥ', 'Ġà¤', '¤', 'à¤ª', 'à¤¸', 'à¥įà¤', 'µ', 'à¥Ģ', 'Ġà¤', 'µ', 'à¤¾à¤', 'Ĺ', 'à¥įà¤', 'µ', 'à¤¿à¤', '¦', 'à¤¾à¤', 'Ĥ', 'Ġà¤', 'µ', 'à¤°', 'à¤®', 'à¥į', 'à¥¤', 'Ġà¤', '¨', 'à¤¾à¤', '°', 'à¤¦', 'à¤Ĥ', 'Ġà¤ª', 'à¤°', 'à¤¿à¤', 'ª', 'à¤ª', 'à¥įà¤°', 'à¤ļ', 'à¥įà¤', 'Ľ', 'Ġà¤', 'µ', 'à¤¾à¤', '²', 'à¥įà¤', '®', 'à¥Ģ', 'à¤ķ', 'à¤¿à¤', '°', 'à¥įà¤', '®', 'à¥ģ', 'à¤¨', 'à¤¿à¤', 'ª', 'à¥ģ', 'à¤', 'Ļ', 'à¥įà¤', 'Ĺ', 'à¤µ', 'à¤®', 'à¥į', 'à¥¥']

English text: The ascetic Vālmīki asked Nārada, the best of sages and foremost of those conversant with words, ever engaged in austerities and Vedic studies.
English tokens (39): ['The', 'Ġasc', 'etic', 'ĠV', 'Äģ', 'lm', 'Ä«', 'ki', 'Ġasked', 'ĠN', 'Äģ', 'r', 'ada', ',', 'Ġthe', 'Ġbest', 'Ġof', 'Ġs', 'ages'

In [5]:
def get_pair(ex):
    if "translation" in ex and isinstance(ex["translation"], dict):
        t = ex["translation"]
        return t["sn"].strip(), t["en"].strip()
    return ex["sn"].strip(), ex["en"].strip()

def is_clean(sa, en):
    if not sa or not en:
        return False
    if len(sa) < 5 or len(en) < 5:
        return False
    if len(sa) > 400 or len(en) > 400:
        return False
    return True

def build_examples(split, n):
    ds = raw[split]
    idxs = list(range(len(ds)))
    random.shuffle(idxs)
    examples = []
    for i in idxs:
        sa, en = get_pair(ds[i])
        if not is_clean(sa, en):
            continue
        examples.append({
            "instruction": "Translate the following Sanskrit verse into English.",
            "input": sa,
            "output": en,
        })
        examples.append({
            "instruction": "Translate the following English sentence into Sanskrit.",
            "input": en,
            "output": sa,
        })
        if len(examples) >= n:
            break
    return examples

N_TRAIN = 6000
N_EVAL  = 200

train_examples = build_examples("train", N_TRAIN)
eval_examples  = build_examples("validation" if "validation" in raw else "test", N_EVAL)

print(len(train_examples), "train examples;", len(eval_examples), "eval examples")
print(train_examples[0])
print(train_examples[1])


6000 train examples; 200 eval examples
{'instruction': 'Translate the following Sanskrit verse into English.', 'input': 'स्तां च ज्ञात्वा परिचर्यां गुरुः सः। तस्मै प्रादात् सद्य एव श्रुतं च भार्यां च वै दुहितरं स्वां सुजाताम्॥', 'output': 'That Brahmana served his preceptor for a long time. Recognising it his preceptor gave him a mastery over the Shastras and also bestowed upon him his own daughter Sujata.'}
{'instruction': 'Translate the following English sentence into Sanskrit.', 'input': 'That Brahmana served his preceptor for a long time. Recognising it his preceptor gave him a mastery over the Shastras and also bestowed upon him his own daughter Sujata.', 'output': 'स्तां च ज्ञात्वा परिचर्यां गुरुः सः। तस्मै प्रादात् सद्य एव श्रुतं च भार्यां च वै दुहितरं स्वां सुजाताम्॥'}


In [6]:
def to_chat_text(ex):
    messages = [
        {"role": "user", "content": f"{ex.get('instruction')}\n\n{ex.get('input')}"},
        {"role": "assistant", "content": ex["output"]},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)

eval_ds_raw = eval_examples
print(to_chat_text(train_examples[0])[:600])


<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
Translate the following Sanskrit verse into English.

स्तां च ज्ञात्वा परिचर्यां गुरुः सः। तस्मै प्रादात् सद्य एव श्रुतं च भार्यां च वै दुहितरं स्वां सुजाताम्॥<|im_end|>
<|im_start|>assistant
That Brahmana served his preceptor for a long time. Recognising it his preceptor gave him a mastery over the Shastras and also bestowed upon him his own daughter Sujata.<|im_end|>



In [7]:
gita_raw = load_with_parquet_fallback("JDhruv14/Bhagavad-Gita_Dataset")
print(gita_raw)
print(gita_raw["train"][0])


README.md:   0%|          | 0.00/952 [00:00<?, ?B/s]

geeta_dataset.csv:   0%|          | 0.00/677k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/701 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['chapter', 'verse', 'sanskrit', 'hindi', 'english', 'transliteration'],
        num_rows: 701
    })
})
{'chapter': 1, 'verse': 1, 'sanskrit': 'धृतराष्ट्र उवाच |धर्मक्षेत्रे कुरुक्षेत्रे समवेता युयुत्सवः |मामकाः पाण्डवाश्चैव किमकुर्वत सञ्जय |', 'hindi': 'धृतराष्ट्र ने कहा—हे संजय! धर्मभूमि कुरुक्षेत्र में एकत्रित, युद्ध की इच्छा वाले मेरे और पाण्डु के पुत्रों ने क्या किया?', 'english': 'Dhritarashtra said: Sanjaya, gathered on the sacred soil of Kurukshetra, eager to fight, what did my children and the children of Pandu do?', 'transliteration': 'dhṛtarāṣṭra uvāca .dharmakṣetre kurukṣetre samavetā yuyutsavaḥ .māmakāḥ pāṇḍavāścaiva kimakurvata sañjaya '}


In [8]:

def build_gita_examples(n=400):
    ds = gita_raw["train"]
    examples = []
    for i in range(len(ds)):
        ex = ds[i]
        sa = (ex.get("sanskrit") or "").strip()
        en = (ex.get("english") or "").strip()
        translit = (ex.get("transliteration") or "").strip()
        if not sa or not en:
            continue

        examples.append({
            "instruction": "Explain what the following Sanskrit verse from the Bhagavad Gita conveys, in English.",
            "input": sa,
            "output": en[:600],
        })

        if translit:
            examples.append({
                "instruction": "Provide the Roman transliteration of the following Sanskrit verse.",
                "input": sa,
                "output": translit[:300],
            })

        if len(examples) >= n:
            break
    return examples

gita_examples = build_gita_examples(n=400)
print(len(gita_examples), "explanation/transliteration examples built from Gita data")
if gita_examples:
    print(gita_examples[0])
    print(gita_examples[1])


400 explanation/transliteration examples built from Gita data
{'instruction': 'Explain what the following Sanskrit verse from the Bhagavad Gita conveys, in English.', 'input': 'धृतराष्ट्र उवाच |धर्मक्षेत्रे कुरुक्षेत्रे समवेता युयुत्सवः |मामकाः पाण्डवाश्चैव किमकुर्वत सञ्जय |', 'output': 'Dhritarashtra said: Sanjaya, gathered on the sacred soil of Kurukshetra, eager to fight, what did my children and the children of Pandu do?'}
{'instruction': 'Provide the Roman transliteration of the following Sanskrit verse.', 'input': 'धृतराष्ट्र उवाच |धर्मक्षेत्रे कुरुक्षेत्रे समवेता युयुत्सवः |मामकाः पाण्डवाश्चैव किमकुर्वत सञ्जय |', 'output': 'dhṛtarāṣṭra uvāca .dharmakṣetre kurukṣetre samavetā yuyutsavaḥ .māmakāḥ pāṇḍavāścaiva kimakurvata sañjaya'}


In [9]:
combined_examples = train_examples + gita_examples
random.shuffle(combined_examples)

train_texts = [to_chat_text(e) for e in combined_examples]
print("Final training set size:", len(train_texts),
      f"({len(train_examples)} translation + {len(gita_examples)} explanation/QA)")


Final training set size: 6400 (6000 translation + 400 explanation/QA)


In [10]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                     "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820


In [11]:
def generate(model, instruction, input_text, max_new_tokens=120):
    messages = [{"role": "user", "content": f"{instruction}\n\n{input_text}"}]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                              pad_token_id=tokenizer.pad_token_id)
    text = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return text.strip()

baseline_outputs = []
for ex in eval_ds_raw[:20]:
    pred = generate(model, ex["instruction"], ex["input"])
    baseline_outputs.append({**ex, "baseline_pred": pred})

for r in baseline_outputs[:5]:
    print("INSTR:", r["instruction"])
    print("INPUT:", r["input"])
    print("GOLD :", r["output"])
    print("PRED :", r["baseline_pred"])
    print("---")


INSTR: Translate the following Sanskrit verse into English.
INPUT: ब्राह्मण उवाच प्राणः प्रालीयत तत: पुनश्च प्रचचार ह। समानश्चाप्युदानश्च वचोऽव्रतां पुनः शुभे॥ न त्वं सर्वमिदं व्याप्य तिष्ठसीह यथा वयम्। न त्वं श्रेष्ठो हि नः प्राण अपानो हि वशे तव। प्रचचार पुनः प्राणस्तमपानोऽभ्यभाषत॥
GOLD : The Brahniana said Prana then became extinct and once more moved about. Then Saman and Udana also, O blessed one, said these words, You do not live here, pervading all this, as we do. You are not the foremost amongst us, O Prana! (Only) Apana is under your dominion! Prana then moved about, and to him Apana spoke.
PRED : Here's an English translation of the given Sanskrit verse:

"O Brahman! The breath is short-lived; again it comes.
With equal breath and with equal speech,
Again he speaks in his own way.
Not you will remain here;
Not you are the best among us;
Your breath is not sufficient for your body."

This verse appears to be describing the transient nature of life and the brevity of human exist

In [14]:
MAX_LEN = 512
BATCH_SIZE = 4
GRAD_ACCUM = 4
MAX_STEPS = 400
LR = 2e-4
LOG_EVERY = 20

pad_id = tokenizer.pad_token_id

def tokenize_text(text):
    ids = tokenizer(text, truncation=True, max_length=MAX_LEN)["input_ids"]
    return ids

tokenized = [tokenize_text(t) for t in train_texts]
print("Tokenized", len(tokenized), "examples. Example length:", len(tokenized[0]))


Tokenized 6400 examples. Example length: 149


In [15]:
def make_batch(examples):
    max_len = max(len(e) for e in examples)
    input_ids = torch.full((len(examples), max_len), pad_id, dtype=torch.long)
    attention_mask = torch.zeros((len(examples), max_len), dtype=torch.long)
    labels = torch.full((len(examples), max_len), -100, dtype=torch.long)
    for i, ids in enumerate(examples):
        input_ids[i, :len(ids)] = torch.tensor(ids, dtype=torch.long)
        attention_mask[i, :len(ids)] = 1
        labels[i, :len(ids)] = torch.tensor(ids, dtype=torch.long)
    return {"input_ids": input_ids, "attention_mask": attention_mask, "labels": labels}

def batch_iterator(data, batch_size):
    idxs = list(range(len(data)))
    while True:
        random.shuffle(idxs)
        for i in range(0, len(idxs) - batch_size + 1, batch_size):
            batch_idxs = idxs[i:i + batch_size]
            yield make_batch([data[j] for j in batch_idxs])

optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=LR)
scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=max(1, int(0.03 * MAX_STEPS)),
    num_training_steps=MAX_STEPS,
)

data_gen = batch_iterator(tokenized, BATCH_SIZE)

model.train()
optimizer.zero_grad()
running_loss = 0.0
step = 0
micro_step = 0

while step < MAX_STEPS:
    batch = next(data_gen)
    batch = {k: v.to(model.device) for k, v in batch.items()}
    outputs = model(**batch)
    loss = outputs.loss / GRAD_ACCUM
    loss.backward()
    running_loss += loss.item()
    micro_step += 1

    if micro_step % GRAD_ACCUM == 0:
        torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()
        step += 1
        if step % LOG_EVERY == 0 or step == 1:
            print(f"step {step}/{MAX_STEPS}  loss={running_loss:.4f}  lr={scheduler.get_last_lr()[0]:.2e}")
        running_loss = 0.0

print("Training complete.")


step 1/400  loss=1.7039  lr=1.67e-05
step 20/400  loss=1.6121  lr=2.00e-04
step 40/400  loss=1.5189  lr=1.97e-04
step 60/400  loss=1.5244  lr=1.93e-04
step 80/400  loss=1.5285  lr=1.85e-04
step 100/400  loss=1.5555  lr=1.76e-04
step 120/400  loss=1.4473  lr=1.64e-04
step 140/400  loss=1.5748  lr=1.51e-04
step 160/400  loss=1.5558  lr=1.36e-04
step 180/400  loss=1.4205  lr=1.21e-04
step 200/400  loss=1.5022  lr=1.05e-04
step 220/400  loss=1.4321  lr=8.87e-05
step 240/400  loss=1.4718  lr=7.28e-05
step 260/400  loss=1.5337  lr=5.77e-05
step 280/400  loss=1.4598  lr=4.36e-05


KeyboardInterrupt: 

In [16]:
model.save_pretrained("./qwen25-1.5b-sanskrit-lora/final_adapter")
tokenizer.save_pretrained("./qwen25-1.5b-sanskrit-lora/final_adapter")
print("Adapter saved.")


Adapter saved.


In [17]:
model.eval()
results = []
for ex in eval_ds_raw[:20]:
    pred = generate(model, ex["instruction"], ex["input"])
    results.append({**ex, "finetuned_pred": pred})

for base, ft in zip(baseline_outputs, results):
    print("INSTR:", base["instruction"])
    print("INPUT:", base["input"])
    print("GOLD :", base["output"])
    print("BASE :", base["baseline_pred"])
    print("FT   :", ft["finetuned_pred"])
    print("=====")


INSTR: Translate the following Sanskrit verse into English.
INPUT: ब्राह्मण उवाच प्राणः प्रालीयत तत: पुनश्च प्रचचार ह। समानश्चाप्युदानश्च वचोऽव्रतां पुनः शुभे॥ न त्वं सर्वमिदं व्याप्य तिष्ठसीह यथा वयम्। न त्वं श्रेष्ठो हि नः प्राण अपानो हि वशे तव। प्रचचार पुनः प्राणस्तमपानोऽभ्यभाषत॥
GOLD : The Brahniana said Prana then became extinct and once more moved about. Then Saman and Udana also, O blessed one, said these words, You do not live here, pervading all this, as we do. You are not the foremost amongst us, O Prana! (Only) Apana is under your dominion! Prana then moved about, and to him Apana spoke.
BASE : Here's an English translation of the given Sanskrit verse:

"O Brahman! The breath is short-lived; again it comes.
With equal breath and with equal speech,
Again he speaks in his own way.
Not you will remain here;
Not you are the best among us;
Your breath is not sufficient for your body."

This verse appears to be describing the transient nature of life and the brevity of human exist

In [18]:
def bleu_chrf(preds, refs):
    bleu = sacrebleu.corpus_bleu(preds, [refs])
    chrf = sacrebleu.corpus_chrf(preds, [refs])
    return bleu.score, chrf.score

def run_eval(model, examples):
    preds = []
    for ex in examples:
        preds.append(generate(model, ex["instruction"], ex["input"]))
    return preds

EVAL_N = 100
eval_subset = eval_ds_raw[:EVAL_N]

print("Generating baseline predictions on eval subset...")
model.disable_adapter_layers()
base_preds_full = run_eval(model, eval_subset)
model.enable_adapter_layers()

print("Generating fine-tuned predictions on eval subset...")
ft_preds_full = run_eval(model, eval_subset)

refs = [ex["output"] for ex in eval_subset]

sa_to_en_idx = [i for i, ex in enumerate(eval_subset) if "into English" in ex["instruction"]]
en_to_sa_idx = [i for i, ex in enumerate(eval_subset) if "into Sanskrit" in ex["instruction"]]

for name, idxs in [("Sanskrit -> English", sa_to_en_idx), ("English -> Sanskrit", en_to_sa_idx)]:
    if not idxs:
        continue
    b_preds = [base_preds_full[i] for i in idxs]
    f_preds = [ft_preds_full[i] for i in idxs]
    r = [refs[i] for i in idxs]
    b_bleu, b_chrf = bleu_chrf(b_preds, r)
    f_bleu, f_chrf = bleu_chrf(f_preds, r)
    print(f"\n{name}  (n={len(idxs)})")

    print(f"  Baseline   BLEU={b_bleu:.2f}  chrF={b_chrf:.2f}")
    print(f"  Fine-tuned BLEU={f_bleu:.2f}  chrF={f_chrf:.2f}")


Generating baseline predictions on eval subset...
Generating fine-tuned predictions on eval subset...

Sanskrit -> English  (n=50)
  Baseline   BLEU=0.31  chrF=24.60
  Fine-tuned BLEU=2.68  chrF=25.63

English -> Sanskrit  (n=50)
  Baseline   BLEU=0.03  chrF=11.08
  Fine-tuned BLEU=0.31  chrF=18.77


In [19]:
def per_example_chrf(preds, refs):
    return [sacrebleu.sentence_chrf(p, [r]).score for p, r in zip(preds, refs)]

scores = per_example_chrf(ft_preds_full, refs)
ranked = sorted(zip(scores, eval_subset, ft_preds_full), key=lambda x: x[0])

print("Worst 10 fine-tuned predictions by chrF:\n")
for score, ex, pred in ranked[:10]:
    print(f"chrF={score:.1f}")
    print("INSTR:", ex["instruction"])
    print("INPUT:", ex["input"])
    print("GOLD :", ex["output"])
    print("PRED :", pred)
    print("---")


Worst 10 fine-tuned predictions by chrF:

chrF=11.4
INSTR: Translate the following Sanskrit verse into English.
INPUT: प्रलब्धश्च हृषीकेशस्तच कर्माविचारितम्। स च मे वचनं ब्रह्मन् कथमेवाभिमन्यते॥
GOLD : The princess Krishna, while standing in the midst of the assembly, wept piteously. Krishna will never forget that act of ours, nor, the deprivation of Yudhishthira by us of his kingdom.
PRED : The Brahman said: O you of great energy, how did this act come to be accomplished?
---
chrF=12.2
INSTR: Translate the following English sentence into Sanskrit.
INPUT: With your power let the sin I have incurred by exterminating the Kashtriyas from wrath be all destroyed. Let also my these lakes becomes Tirthas, celebrated all over the earth.
GOLD : यच्च रोषाभिभूतेन क्षत्रमुत्सादितं मया। ततश्च पापान्मुच्येयं युष्माकं तेजसाप्यहम्॥ ह्रदाश्च तीर्थभूता मे भवेयुर्भुवि विश्रुताः।
PRED : कश्त्रियान् वीरं शृणु मे कथमेको हतः। अन्यास्य च स्थानं प्रयच्छसि देहिना भव॥
---
chrF=12.9
INSTR: Translate the following

In [20]:
def content_words(text):
    text = re.sub(r"[।॥,.!?\"\'()\[\]]", " ", text)
    return set(w for w in text.split() if len(w) > 1)

def overlap_ratio(pred, source, gold):
    pred_words = content_words(pred)
    if not pred_words:
        return 0.0
    context_words = content_words(source) | content_words(gold)
    overlap = pred_words & context_words
    return len(overlap) / len(pred_words)

hallucination_flags = []
for ex, pred in zip(eval_subset, ft_preds_full):
    ratio = overlap_ratio(pred, ex["input"], ex["output"])
    hallucination_flags.append({
        "instruction": ex["instruction"],
        "input": ex["input"],
        "gold": ex["output"],
        "pred": pred,
        "overlap_ratio": ratio,
    })

hallucination_flags.sort(key=lambda x: x["overlap_ratio"])

print("Top 10 predictions most likely to contain hallucinated (ungrounded) content:\n")
for h in hallucination_flags[:10]:
    print(f"overlap_ratio={h.get('overlap_ratio'):.2f}")
    print("INSTR:", h["instruction"])
    print("INPUT:", h["input"])
    print("GOLD :", h["gold"])
    print("PRED :", h["pred"])
    print("---")

import pandas as pd
pd.DataFrame(hallucination_flags).to_csv("hallucination_check.csv", index=False)
print("\nSaved hallucination_check.csv")


Top 10 predictions most likely to contain hallucinated (ungrounded) content:

overlap_ratio=0.00
INSTR: Translate the following English sentence into Sanskrit.
INPUT: If the science of punishment disappears, the Vedas will be lost. All these scriptures also describing the duties of men become lost. Indeed, if these ancient duties the Kshatriyas be abandoned, all the duties of all the modes of life, become lost.
GOLD : मज्जेत् त्रयी दण्डनीतौ हतायां सर्वे धर्माः प्रक्षयेयुर्विबुद्धाः। सर्वे धर्माश्चाश्रमाणां हताः स्युः क्षात्रे त्यक्ते राजधर्मे पुराणे॥
PRED : यदि तु प्रतिष्ठांस्तथैव कृत्वा संशयः परम्। यदि न चेदं रक्षन्ते धर्मो भागं श्रुतिस्तथा॥ अभिप्राप्तं बहुधा धर्माणां जग्मुर्मन्�
---
overlap_ratio=0.00
INSTR: Translate the following English sentence into Sanskrit.
INPUT: O foremost of men, the degraded caste, the Śūdras, during your rege me, have engaged in austere penances. And in the Kali Yuga asceticism shall be established in the Sūdras.
GOLD : हीनवर्णो नृपश्रेष्ठ तप्यते सुमहत्तपः

In [21]:
import pandas as pd

df = pd.DataFrame({
    "instruction": [ex["instruction"] for ex in eval_subset],
    "input": [ex["input"] for ex in eval_subset],
    "gold": refs,
    "baseline_pred": base_preds_full,
    "finetuned_pred": ft_preds_full,
    "chrf": scores,
})
df.to_csv("eval_results.csv", index=False)
print("Saved eval_results.csv -", len(df), "rows")
df.head()


Saved eval_results.csv - 100 rows


,instruction,input,gold,baseline_pred,finetuned_pred,chrf
0,Translate the following Sanskrit verse into En...,ब्राह्मण उवाच प्राणः प्रालीयत तत: पुनश्च प्रचच...,The Brahniana said Prana then became extinct a...,Here's an English translation of the given San...,The Brahmana said : The breath is born from Pr...,26.045811
1,Translate the following English sentence into ...,The Brahniana said Prana then became extinct a...,ब्राह्मण उवाच प्राणः प्रालीयत तत: पुनश्च प्रचच...,Here's the translation of the given English se...,ब्रह्मण उवाच प्राणं ततोऽनिवृद्धमभिप्राप्य समान...,20.700490
2,Translate the following Sanskrit verse into En...,मज्जेत् त्रयी दण्डनीतौ हतायां सर्वे धर्माः प्र...,"If the science of punishment disappears, the V...",Here is the translation of the Sanskrit verse ...,The three kings were slain by the arrows of Ja...,20.933207
3,Translate the following English sentence into ...,"If the science of punishment disappears, the V...",मज्जेत् त्रयी दण्डनीतौ हतायां सर्वे धर्माः प्र...,"यदि शास्त्र की विजेता नहोती हो, ये वैज्ञाणांमु...",यदि तु प्रतिष्ठांस्तथैव कृत्वा संशयः परम्। यदि...,20.638525
4,Translate the following Sanskrit verse into En...,अस्याः शुल्कं राज्यमपि ध्रुवम्। किं पुनः श्याम...,For her beauty do you accept. Rulers of men wi...,Here is the translation of the Sanskrit verse ...,The king of the Shulkas and the king of the Ha...,19.699071


In [22]:
ayurveda_passage = "अथ आयुर्वेदः नाम शास्त्रम् आयुर्वेदः नाम आयुः वेदयति इति आयुर्वेदः आयुः नाम शरीर इन्द्रिय सत्त्व आत्म संयोगः तस्य हिताहितं सुखं दुःखम् आयुः तस्य हितं च अहितं च मानम् च तच् च यत्र उक्तं तत् आयुर्वेदः"

print("Translation:")
print(generate(model, "Translate the following Sanskrit verse into English.", ayurveda_passage, max_new_tokens=150))
print()
print("Explanation:")
print(generate(model, "Explain the meaning of the following Sanskrit verse in simple terms.", ayurveda_passage, max_new_tokens=200))


Translation:
The Vedas are called Ayurveda and Ayurveda is called Ayu. The body is called Ayu; mind is called Ayu; soul is called Ayu; senses are called Ayu; Indra is called Ayu; the elements of earth, water, fire and air are all called Ayu. The good or bad effects of actions depend on Ayu. Ayu is said to be beneficial or unbeneficial according as it is used for good or evil.

Explanation:
The Vedas are called Ayurveda and Ayurveda is called Ayu. Ayu means body; Indra means senses; Satva means mind; Aham means self; Sankya means union; that which is beneficial to one's own welfare is said to be Ayurveda.
